[![Fixel Algorithms](https://i.imgur.com/AqKHVZ0.png)](https://fixelalgorithms.gitlab.io)

# AI Program

## Deep Learning - Computer Vision - LeNet

> Notebook by:
> - Royi Avital RoyiAvital@fixelalgorithms.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 01/08/2026 | Royi Avital | First version                                                      |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FixelAlgorithmsTeam/FixelCourses/blob/master/AIProgram/2024_02/0091DeepLearningPreTrainedModels.ipynb)

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Machine Learning

# Deep Learning
import torch
import torch.nn            as nn
import torch.nn.functional as F
import torchinfo
from torchmetrics.classification import MulticlassAccuracy
import torchvision
from torchvision.transforms import v2 as TorchVisionTrns
import torchvista

# Image Processing & Computer Vision

# Miscellaneous
import math
import os
from platform import python_version
import random

# Typing
from typing import Any, Callable, Dict, Generator, List, Optional, Self, Set, Tuple, Union
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

## Notations

* <font color='red'>(**?**)</font> Question to answer interactively.
* <font color='blue'>(**!**)</font> Simple task to add code for the notebook.
* <font color='green'>(**@**)</font> Optional / Extra self practice.
* <font color='brown'>(**#**)</font> Note / Useful resource / Food for thought.

Code Notations:

```python
someVar    = 2; #<! Notation for a variable
vVector    = np.random.rand(4) #<! Notation for 1D array
mMatrix    = np.random.rand(4, 3) #<! Notation for 2D array
tTensor    = np.random.rand(4, 3, 2, 3) #<! Notation for nD array (Tensor)
tuTuple    = (1, 2, 3) #<! Notation for a tuple
lList      = [1, 2, 3] #<! Notation for a list
dDict      = {1: 3, 2: 2, 3: 1} #<! Notation for a dictionary
oObj       = MyClass() #<! Notation for an object
dfData     = pd.DataFrame() #<! Notation for a data frame
dsData     = pd.Series() #<! Notation for a series
hObj       = plt.Axes() #<! Notation for an object / handler / function handler
```

### Code Exercise

 - Single line fill

```python
valToFill = ???
```

 - Multi Line to Fill (At least one)

```python
# You need to start writing
?????
```

 - Section to Fill

```python
#===========================Fill This===========================#
# 1. Explanation about what to do.
# !! Remarks to follow / take under consideration.
mX = ???

?????
#===============================================================#
```

In [ ]:
# Configuration
# %matplotlib inline

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
# sns.set_theme() #>! Apply SeaBorn theme

runInGoogleColab = 'google.colab' in str(get_ipython())

# Improve performance by benchmarking
torch.backends.cudnn.benchmark = True

# Reproducibility (Per PyTorch Version on the same device)
# torch.manual_seed(seedNum)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False #<! Makes things slower

In [ ]:
# Constants

FIG_SIZE_DEF    = (8, 8)
ELM_SIZE_DEF    = 50
CLASS_COLOR     = ('b', 'r')
EDGE_COLOR      = 'k'
MARKER_SIZE_DEF = 10
LINE_WIDTH_DEF  = 2

PROJECT_NAME     = 'FixelCourses'
DATA_FOLDER_NAME = 'DataSets'
BASE_FOLDER_PATH = os.getcwd()[:(len(os.getcwd()) - (os.getcwd()[::-1].lower().find(PROJECT_NAME.lower()[::-1])))]
DATA_FOLDER_PATH = os.path.join(BASE_FOLDER_PATH, DATA_FOLDER_NAME)

TENSOR_BOARD_BASE   = 'TB'

D_CLASSES = {ii: str(ii) for ii in range(10)}
L_CLASSES = [ii for ii in range(len(D_CLASSES))]

In [ ]:
# Download Auxiliary Modules for Google Colab
if runInGoogleColab:
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataManipulation.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DataVisualization.py
    !wget https://raw.githubusercontent.com/FixelAlgorithmsTeam/FixelCourses/master/AIProgram/2024_02/DeepLearningPyTorch.py

In [ ]:
# Courses Packages

from DeepLearningPyTorch import TrainModel

In [ ]:
# General Auxiliary Functions


## LeNet

The _LeNet_ family of models are a foundational milestone in the history of _Deep Learning_ and _Computer Vision_.  
It was the first _Convolutional Neural Network_ to successfully combine *Convolutional Layers* with _Backpropagation_ training,
demonstrating that learned hierarchical feature representations could effectively solve real world image classification tasks.

This notebook reproduce a similar model to LeNet of 1990 as given by the paper: [Handwritten Digit Recognition with a Back Propagation Network](https://dl.acm.org/doi/10.5555/2969830.2969879).


### Historical Context

Before LeNet, image classification relied on a hand crafted features.  
The key innovation of LeNet was the integration of:

1. **Convolutional Layers**  
   Local feature extraction through learned filters.
2. **Pooling Layers**  
   Downsampling to provide translation invariance and computational efficiency.
3. **Fully Connected Layers**  
   A feature combination for final classification.
4. **Backpropagation Training**  
   Efficient learning of all layers simultaneously.

This architecture proved that _Convolutional Neural Networks_ could learn effective representations automatically,
eliminating the need for manual feature engineering.

### LeNet Versions

Several versions of LeNet evolved over time, each improving upon the previous:

* **LeNet 1** (1991)  
  The original version with basic convolutional and pooling layers.  
  Designed specifically for handwritten digit recognition.

* **LeNet 4** (1995)  
  An intermediate variant with increased depth and capacity.

* **LeNet 5** (1998)  
  The most well known version, which achieved state of the art performance on the MNIST dataset.  
  It consists of: 2 convolutional layers, 3 fully connected layers  and uses the _LeCun TanH activation_ function.  
  Trained on the MNIST dataset, it achieved recognition accuracy exceeding 99% and was deployed in commercial check reading systems.

</br>

* <font color='brown'>(**#**)</font> The original LeNet paper: [Gradient based Learning Applied to Document Recognition](https://ieeexplore.ieee.org/document/726791). 
* <font color='brown'>(**#**)</font> LeNet achieved 99.2% accuracy on MNIST, a result that remained state of the art for many years.  
  Actually, to these days, it is hard to beat it with the same number of parameters as most modern techniques are focused on making models deeper.
* <font color='brown'>(**#**)</font> The term "_Convolutional Neural Network_" and "CNN" became standard terminology following LeNet's success.

In [ ]:
# Parameters

# Data

# Model

# Training
batchSize = 128
numEpochs = 30
numWork   = 4

# Visualization
numImg = 3 * 3 #<! A squared number

## Generate / Load Data



In [ ]:
# Load Data

dsTrain = torchvision.datasets.MNIST(root = DATA_FOLDER_PATH, train = True, download = True)
dsVal   = torchvision.datasets.MNIST(root = DATA_FOLDER_PATH, train = False, download = True)

numSamplesTrain = len(dsTrain)
numSamplesVal   = len(dsVal)

numCls = len(dsTrain.classes)

mI, valY = dsTrain[0]
imgSize  = mI.size[0] #<! Assuming square images

# Data Set Element:
print(f'Number of Training Samples: {numSamplesTrain}')
print(f'Number of Validation Samples: {numSamplesVal}')
print(f'Image Type: {type(mI)}')
print(f'Image Size: {mI.size}')

### Plot the Data

In [ ]:
# Plot the Data

hF, vHa = plt.subplots(nrows = int(math.sqrt(numImg)), ncols = int(math.sqrt(numImg)), figsize = (5, 5))

for ii, hA in enumerate(vHa.flat):
    imgIdx = random.randint(0, numSamplesTrain - 1)
    mI, valY = dsTrain[imgIdx]
    hA.imshow(mI, 'gray')
    hA.tick_params(axis = 'both', left = False, top = False, right = False, bottom = False, 
                   labelleft = False, labeltop = False, labelright = False, labelbottom = False)
    hA.set_title(f'Label: {valY}')
    hA.grid(False)

## Define the Model

This section defines the architecture of the LeNet 1990 model.

* <font color='brown'>(**#**)</font> A deeper analysis of the architecture of LeNet is done in [Andrej Karpathy - Deep Neural Nets: 33 Years Ago and 33 Years from Now](https://karpathy.github.io/2022/03/14/lecun1989).

### LeNet (1990) Architecture Diagram

```mermaid
flowchart TB
    IN[1 x 28 x 28]

    subgraph R1[ ]
        direction LR
        C1["Conv 1<br/>5 x 5, stride = 1"]
        T1[TanH]
        P1["AvgPool<br/>2 x 2, stride = 2"]
    end

    subgraph R2[ ]
        direction LR
        C2["Conv 2<br/>5 x 5, stride = 1"]
        T2[TanH]
        P2["AvgPool<br/>2 x 2, stride = 2"]
    end

    FC[FC]
    Y[ŷ]

    IN --> C1
    C1 -->|4 x 24 x 24| T1
    T1 --> P1
    P1 -->|4 x 12 x 12| C2

    C2 -->|12 x 8 x 8| T2
    T2 --> P2
    P2 -->|12 x 4 x 4 = 192| FC

    FC -->|10 classes| Y

    classDef conv fill:#1f78a8,color:#ffffff,stroke:#e6eef5,stroke-width:1px;
    classDef act  fill:#b22222,color:#ffffff,stroke:#e6eef5,stroke-width:1px;
    classDef pool fill:#5b3a82,color:#ffffff,stroke:#e6eef5,stroke-width:1px;
    classDef head fill:#2c2f36,color:#f5e65a,stroke:#d8d8d8,stroke-width:1px;
    classDef fc   fill:#1f78a8,color:#ffffff,stroke:#e6eef5,stroke-width:1px;
    classDef out  fill:#2c2f36,color:#f5e65a,stroke:#d8d8d8,stroke-width:1px;

    class C1,C2 conv;
    class T1,T2 act;
    class P1,P2 pool;
    class IN head;
    class FC fc;
    class Y out;
```

In [ ]:
# The Model Class

class LeNet(nn.Module):
    def __init__(self, numCls: int = 10) -> None:
        super().__init__()
        #===========================Fill This===========================#
        # 1. Define the compute graph components of the model.
        # !! You should use `torch.flatten()` to flatten the features map from 2D to 1D before the fully connected layer.
        self.conv1 = ???
        self.pool1 = ???
        self.conv2 = ???
        self.pool2 = ???
        self.fc1   = ???
        #===============================================================#

    def forward(self, tX: Tensor) -> Tensor:
        #===========================Fill This===========================#
        # 1. Implement the compute graph of the model.
        # !! You should use `torch.flatten()` to flatten the features map from 2D to 1D before the fully connected layer.
        ?????
        #===============================================================#

        return tX

### Model Visualization

This section shows the models using `torchinfo` and `torchvista`.  

* <font color='brown'>(**#**)</font> Since the information is limited to the architecture, no need to load the pre trained weights.

In [ ]:
# Model Realization

oModel = LeNet(numCls = numCls)

In [ ]:
# Summary of the Model
torchinfo.summary(oModel, (batchSize, 1, imgSize, imgSize), col_names = ['kernel_size', 'output_size', 'num_params'], device = 'cpu')

In [ ]:
# Visualize the Model
tX = torch.randn(batchSize, 1, imgSize, imgSize)
torchvista.trace_model(oModel.eval(), tX)

## Train the Model

This section trains the model.

In [ ]:
# Update Transforms

oDataTrns = TorchVisionTrns.Compose([
    TorchVisionTrns.ToImage(),
    TorchVisionTrns.ToDtype(torch.float32, scale = True),
])

# Update the DS transformer
dsTrain.transform   = oDataTrns
dsVal.transform     = oDataTrns

In [ ]:
# Data Loaders

dlTrain = torch.utils.data.DataLoader(dsTrain, shuffle = True, batch_size = 1 * batchSize, num_workers = numWork, persistent_workers = True)
dlVal   = torch.utils.data.DataLoader(dsVal, shuffle = False, batch_size = 2 * batchSize, num_workers = numWork, persistent_workers = True)

In [ ]:
# Run Device

runDevice = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #<! The 1st CUDA device
oModel    = oModel.to(runDevice) #<! Transfer model to device

print(f'Run Device: {runDevice}')

In [ ]:
# Loss and Score Function

hL = nn.CrossEntropyLoss()
hS = MulticlassAccuracy(num_classes = numCls, average = 'micro')
hL = hL.to(runDevice) #<! Not required!
hS = hS.to(runDevice)

* <font color='brown'>(**#**)</font> The averaging mode `macro` averages samples per class and average the result of each class.
* <font color='brown'>(**#**)</font> The averaging mode `micro` averages all samples.
* <font color='red'>(**?**)</font> Given 8 samples of class `A` with 6 predictions being correct and 2 samples of class `B` with 1 being correct.  
  What will be the _macro average_? What will be the _micro average_?

In [ ]:
# Define Optimizer

oOpt = torch.optim.AdamW(oModel.parameters(), lr = 1e-3, betas = (0.9, 0.99), weight_decay = 1e-3) #<! Define optimizer

In [ ]:
# Define Scheduler

oSch = torch.optim.lr_scheduler.OneCycleLR(oOpt, max_lr = 5e-3, total_steps = numEpochs)

In [ ]:
# Train Model

oModel, lTrainLoss, lTrainScore, lValLoss, lValScore, lLearnRate = TrainModel(oModel, dlTrain, dlVal, oOpt, numEpochs, hL, hS, oSch = oSch)

* <font color='brown'>(**#**)</font> The original 1990 model ran on [SPARCstation 10](https://en.wikipedia.org/wiki/SPARCstation_10) which cost ~20,000 [$].  
The training of 30 epochs took ~3 days. Inference of a single image took 0.015 [Sec].

In [ ]:
# Plot Training Phase

hF, vHa = plt.subplots(nrows = 1, ncols = 3, figsize = (14, 5))
vHa = np.ravel(vHa)

hA = vHa[0]
hA.plot(lTrainLoss, lw = 2, label = 'Train')
hA.plot(lValLoss, lw = 2, label = 'Validation')
hA.set_title('Binary Cross Entropy Loss')
hA.set_xlabel('Epoch')
hA.set_ylabel('Loss')
hA.legend()

hA = vHa[1]
hA.plot(lTrainScore, lw = 2, label = 'Train')
hA.plot(lValScore, lw = 2, label = 'Validation')
hA.set_title('Accuracy Score')
hA.set_xlabel('Epoch')
hA.set_ylabel('Score')
hA.legend()

hA = vHa[2]
hA.plot(lLearnRate, lw = 2)
hA.set_title('Learn Rate Scheduler')
hA.set_xlabel('Epoch')
hA.set_ylabel('Learn Rate');

* <font color='green'>(**@**)</font> Implement LeNet 1998 (See [Wikipedia - LeNet](https://en.wikipedia.org/wiki/LeNet)).  
  The model input is `(1, 32, 32)`. Using `torchvision.transforms.v2.CenterCrop` / `torchvision.transforms.v2.Resize` might be useful.